# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Code from Week 1 & 2

In [ ]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
from io import BytesIO
from PIL import Image
import sqlite3
import json

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
# AI setup

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
# DB setup
DB = "students.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS students (name TEXT PRIMARY KEY, age REAL, subject TEXT)')
    conn.commit()

In [ ]:
# DB tool functions

def get_student(name, age):
    if not name:
        return "Error: name is required"
    elif not age:
        return "Error: age is required"
    print(f"DATABASE TOOL CALLED: Getting student info for {name}, age: {age}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT subject FROM students WHERE name = ? AND age = ?', (name.lower(), age))
        result = cursor.fetchone()
        return f"{name} who is {age} years old, is currently studying {result[0]}" if result else f"No subject info for {name}, {age}."
    
# could have better table logic, being name and age 
def register_student(name, age, subject):
    if not name:
        return "Error: name is required"
    elif not age:
        return "Error: age is required"
    elif not subject:
        return "Error: subject is required"
    print(f"DATABASE TOOL CALLED: Registering student info for {name}, age: {age}, subject: {subject}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            INSERT INTO students (name, age, subject)
            VALUES (?, ?, ?)
            ON CONFLICT(name)
            DO UPDATE SET
                age = excluded.age,
                subject = excluded.subject
            """,
            (name.lower(), age, subject)
        )
        conn.commit()
        return f"Student: {name}, {age} years old is registered to study {subject}."

In [ ]:
get_student_function = {
    "name": "get_student",
    "description": "Get the subject that a student with a given name and age is currently studying.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name of the student."
            },
            "age": {
                "type": "number",
                "description": "The age of the student."
            }
        },
        "required": ["name", "age"],
        "additionalProperties": False
    }
}
register_student_function = {
    "name": "register_student",
    "description": "Register a student's name, age and the subject they are studying; or update any of the current student's info.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name of the student."
            },
            "age": {
                "type": "number",
                "description": "The age of the student.",
            },
            "subject": {
                "type": "string",
                "description": "The age of the student."
            }
        },
        "required": ["name", "age", "subject"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": get_student_function},{"type": "function", "function": register_student_function}]

In [ ]:
def has_required_args(tool_name, arguments):
    if tool_name == "get_student":
        return bool(arguments.get("name")) and arguments.get("age") is not None

    if tool_name == "register_student":
        return (
            bool(arguments.get("name")) and
            arguments.get("age") is not None and
            bool(arguments.get("subject"))
        )

    return False


In [ ]:
# TODO: change cities for something more relevant for image generation  
def handle_tool_calls(message):
    responses = []

    for tool_call in message["tool_calls"]:
        tool_name = tool_call["function"]["name"]
        raw_args = tool_call["function"]["arguments"]

        try:
            arguments = json.loads(raw_args) if raw_args.strip() else {}
        except json.JSONDecodeError:
            arguments = {}

        if not has_required_args(tool_name, arguments):
            continue

        if tool_name == "get_student":
            result = get_student(
                arguments.get("name"),
                arguments.get("age")
            )

        elif tool_name == "register_student":
            result = register_student(
                arguments.get("name"),
                arguments.get("age"),
                arguments.get("subject")
            )

        else:
            continue

        responses.append({
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "content": str(result)
        })

    return responses


In [ ]:
system_prompt = """
You're a helpful tutor that explains in detail and in simple terms.
You adapt your teaching style to the age of the person and the subject
they are currently learning. The student can only ask questions about
the subject they are learning. If they ask about another subject,
redirect them back to the subject they are registered with.
"""

In [ ]:
# # TODO: change this for other tutoring image generation
# def artist(city):
#     image_response = openai.images.generate(
#             model="dall-e-3",
#             prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
#             size="1024x1024",
#             n=1,
#             response_format="b64_json",
#         )
#     image_base64 = image_response.data[0].b64_json
#     image_data = base64.b64decode(image_base64)
#     return Image.open(BytesIO(image_data))

In [ ]:
def chat_gpt(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools, stream=True)
    # cities = []
    # image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # responses, cities = handle_tool_calls(message)
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # if cities:
    #     image = artist(cities[0])
    
    return history

def chat_ollama(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, tools=tools)
    # cities = []
    # image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # if cities:
    #     image = artist(cities[0])
    
    return history

    # return history, image

In [ ]:
def chat_gpt_stream(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history

    while True:
        stream = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            tools=tools,
            stream=True
        )

        assistant_message = {"role": "assistant", "content": ""}
        tool_calls = {}

        for chunk in stream:
            choice = chunk.choices[0]
            delta = choice.delta

            # Stream assistant text
            if delta.content:
                assistant_message["content"] += delta.content
                yield history + [assistant_message]

            # Collect tool calls
            if delta.tool_calls:
                for call in delta.tool_calls:
                    call_id = call.id

                    if call_id not in tool_calls:
                        tool_calls[call_id] = {
                            "id": call_id,
                            "type": "function",
                            "function": {
                                "name": call.function.name,
                                "arguments": ""
                            }
                        }

                    if call.function.arguments:
                        tool_calls[call_id]["function"]["arguments"] += call.function.arguments

            if choice.finish_reason:
                break

        # No tools → normal stop
        if not tool_calls:
            messages.append(assistant_message)
            history.append(assistant_message)
            return

        # Execute tools
        messages.append({
            "role": "assistant",
            "tool_calls": list(tool_calls.values())
        })

        tool_responses = handle_tool_calls(
            {"tool_calls": list(tool_calls.values())}
        )

        messages.extend(tool_responses)


In [ ]:
def stream_model(history, model):
    if model=="GPT":
        result = chat_gpt_stream(history)
    elif model=="Ollama":
        result = chat_ollama(history)
    else:
        raise ValueError("Unknown model")
    # return result
    yield from result

In [ ]:
# TODO: streaming

# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        # image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        model_selector = gr.Dropdown(["GPT", "Ollama"], label="Select model", value="GPT")
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Tutor:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        stream_model, inputs=[chatbot, model_selector], outputs=[chatbot]
    )

ui.launch(inbrowser=True)
# ui.launch(inbrowser=True, auth=("tutoring", "tutor123"))